# 002 - Fase 2

In [3]:
import pandas as pd
import numpy as np
import polars as pl

In [4]:
df_customers = pd.read_csv('../data/sales_customers.csv')
df_employees = pd.read_csv('../data/sales_employees.csv')
df_orders = pd.read_csv('../data/sales_orders.csv')
df_orderarchive = pd.read_csv('../data/sales_ordersarchive.csv')
df_products = pd.read_csv('../data/sales_products.csv')

pl_customers = pl.read_csv('../data/sales_customers.csv')
pl_employees = pl.read_csv('../data/sales_employees.csv')
pl_orders = pl.read_csv('../data/sales_orders.csv')
pl_orderarchive = pl.read_csv('../data/sales_ordersarchive.csv')
pl_products = pl.read_csv('../data/sales_products.csv')

# 📝 Tu Reto de Fase 2 (Nivel Iniciación):
Usando la tabla sales_customers.csv, necesito un reporte de:

Clientes que sean de 'Germany'.

O que tengan un puntaje (score) mayor a 500.

Selecciona solo firstname y country.

¿Cómo harías esta Traducción Triple? (Pista: Para el "O" se usa OR en SQL, y el símbolo | en Pandas/Polars).

```SQL
SELECT
    firstname,
    country
FROM sales.customers
WHERE LOWER(country) = 'germany'
OR score > 500;
```

In [7]:
df_c = df_customers.copy()

df_c2 = df_c[
    (df_c['country'].str.lower() == 'germany') |
    (df_c['score'] > 500)
]

respuesta  = df_c2[['firstname','country']]

respuesta

,firstname,country
0,Jossef,Germany
1,Kevin,USA
2,Mary,USA
3,Mark,Germany


In [8]:
# Polars

respuesta_pl = pl_customers.filter(
    (pl.col('country').str.to_lowercase()=='germany') |
    (pl.col('score') > 500)
).select([
    pl.col('firstname'),
    pl.col('country')
])

respuesta_pl

firstname,country
str,str
"""Jossef""","""Germany"""
"""Kevin""","""USA"""
"""Mary""","""USA"""
"""Mark""","""Germany"""


# 📝 Escenario de Práctica: "Auditoría de Clientes y Calidad de Datos"
Usando tu tabla sales_customers.csv, el departamento de marketing necesita un reporte con las siguientes condiciones:

1. Buscamos clientes de USA o Germany (Usa el operador IN).
2. Y que su puntaje (score) esté entre 100 y 500 inclusive (Usa el operador BETWEEN).
3. O que directamente no tengan puntaje (clientes nuevos con score nulo).

```SQL
SELECT
    firstname,
    lastname,
    country,
    score
FROM sales.customers
WHERE (
  LOWER(country) in ('usa', 'germany')
  AND (score BETWEEN 100 AND 500 OR score IS NULL)
);
```

In [6]:
df_c = df_customers.copy()

respuesta = df_c[
    (df_c['country'].str.lower().isin(['usa','germany'])) &
    ((df_c['score'].between(100,500)) | (df_c['score'].isna()))
]

respuesta = respuesta[['firstname','lastname','country','score']]

respuesta

,firstname,lastname,country,score
0,Jossef,Goldberg,Germany,350.0
3,Mark,Schwarz,Germany,500.0
4,Anna,Adams,USA,NaN


In [7]:
respuesta = pl_customers.filter(
    (pl.col('country').str.to_lowercase().is_in(['usa','germany'])) &
    (
        pl.col('score').is_between(100,500) |
        pl.col('score').is_null()
    )
).select([
    pl.col('firstname'),
    pl.col('lastname'),
    pl.col('country'),
    pl.col('score')
])

respuesta

firstname,lastname,country,score
str,str,str,i64
"""Jossef""","""Goldberg""","""Germany""",350
"""Mark""","""Schwarz""","""Germany""",500
"""Anna""","""Adams""","""USA""",null


# 📝 Escenario: "Limpieza de Inventario y Auditoría de Ventas"
Vamos a usar la tabla sales_orders.csv. El equipo de finanzas necesita identificar órdenes específicas para un reporte de impuestos.

La condición es:

Órdenes cuyo orderstatus sea 'Shipped' o 'Delivered' (Usa IN).

Y que la cantidad (quantity) sea exactamente 1 o 2.

Y que además cumplan una de estas dos sub-condiciones:

Que las ventas (sales) estén entre 10 y 50 inclusive (Usa BETWEEN).

O que la dirección de facturación (billaddress) esté vacía (Usa IS NULL o detecta si es un string vacío '').

```SQL
SELECT
    *
FROM sales.orders
WHERE (LOWER(orderstatus) IN ('shipped','delivered'))
AND (quantity IN ( 1,2))
AND ((sales BETWEEN 10 AND 50) OR (billaddress IS NULL OR billaddress = ''));
```

In [ ]:
df_o = df_orders.copy()

respuesta = df_o[
    (df_o['orderstatus'].str.lower().isin(['shipped','delivered'])) &
    (df_o['quantity'].isin([1,2])) &
    (
        (df_o['sales'].between(10,50)) |
        (df_o['billaddress'].isna()) |
        (df_o['billaddress'] == '')
    )
]

In [ ]:
pl_o = pl_orders

respuesta = pl_o.filter(
    (pl.col('orderstatus').str.to_lowercase().is_in(['shipped','delivered'])) &
    (pl.col('quantity').is_in([1,2])) &
    (
        (pl.col('sales').is_between(10,50)) |
        (pl.col('billaddress').is_null()) |
        (pl.col('billaddress') == '')
    )
)

respuesta

orderid,productid,customerid,salespersonid,orderdate,shipdate,orderstatus,shipaddress,billaddress,quantity,sales,creationtime
i64,i64,i64,i64,str,str,str,str,str,i64,i64,str
1,101,2,3,"""2025-01-01""","""2025-01-05""","""Delivered""","""9833 Mt. Dias Blv.""","""1226 Shoe St.""",1,10,"""2025-01-01 12:34:56.000000"""
2,102,3,3,"""2025-01-05""","""2025-01-10""","""Shipped""","""250 Race Court""",null,1,15,"""2025-01-05 23:22:04.000000"""
3,101,1,5,"""2025-01-10""","""2025-01-25""","""Delivered""","""8157 W. Book""","""8157 W. Book""",2,20,"""2025-01-10 18:24:08.000000"""
4,105,1,3,"""2025-01-20""","""2025-01-25""","""Shipped""","""5724 Victory Lane""","""""",2,60,"""2025-01-20 05:50:33.000000"""
5,104,2,5,"""2025-02-01""","""2025-02-05""","""Delivered""",null,null,1,25,"""2025-02-01 14:02:41.000000"""
6,104,3,5,"""2025-02-05""","""2025-02-10""","""Delivered""","""1792 Belmont Rd.""",null,2,50,"""2025-02-06 15:34:57.000000"""
7,102,1,1,"""2025-02-15""","""2025-02-27""","""Delivered""","""136 Balboa Court""","""""",2,30,"""2025-02-16 06:22:01.000000"""
9,101,2,3,"""2025-03-10""","""2025-03-15""","""Shipped""","""3768 Door Way""","""""",2,20,"""2025-03-10 12:59:04.000000"""


# 📝 Escenario: "Categorización de Salarios y Género"
Imagina que el departamento de Recursos Humanos necesita un reporte de la tabla sales_employees.csv. Quieren una nueva columna llamada Nivel_Salarial y un filtro específico.

**Las condiciones son:**

1. Filtro: Solo queremos empleados del departamento de 'Sales' o 'Marketing'.
2. Nueva Columna (Nivel_Salarial):
    - Si el salario es mayor a 70,000, la etiqueta es 'Alto'.
    - Si el salario está entre 50,000 y 70,000 (inclusive), la etiqueta es 'Medio'.
    - Para cualquier otro caso, la etiqueta es 'Bajo'.

```SQL
SELECT
    employeeid,
    firstname,
    salary,
    CASE
        WHEN salary > 70000 THEN 'Alto'
        WHEN salary BETWEEN 50000 AND 70000 THEN 'Medio'
        ELSE 'Bajo'
    END AS nivel_salario
FROM sales.employees
WHERE LOWER(department) IN ('marketing','sales');
```

In [20]:
df_e = df_employees.copy()
df_e = df_e[df_e['department'].str.lower().isin(['marketing','sales'])]

condiciones = [
    (df_e['salary'] > 70000),
    (df_e['salary'].between(50000, 70000))
]

elecciones = ['Alto','Medio']

df_e['nivel_salario'] = np.select(condiciones, elecciones, default='Bajo')

respuesta = df_e[['employeeid','firstname','salary','nivel_salario']]

respuesta




,employeeid,firstname,salary,nivel_salario
0,1,Frank,55000,Medio
1,2,Kevin,65000,Medio
2,3,Mary,75000,Alto
3,4,Michael,90000,Alto
4,5,Carol,55000,Medio


In [22]:
respuesta_pl = pl_employees.filter(
    pl.col('department').str.to_lowercase().is_in(['marketing','sales'])
).with_columns(
    pl.when(pl.col('salary')>70000).then(pl.lit('Alto'))
    .when(pl.col('salary').is_between(50000,70000)).then(pl.lit('Medio'))
    .otherwise(pl.lit('Bajo'))
    .alias('nivel_salario')
).select(['employeeid','firstname','salary','nivel_salario'])

respuesta_pl

employeeid,firstname,salary,nivel_salario
i64,str,i64,str
1,"""Frank""",55000,"""Medio"""
2,"""Kevin""",65000,"""Medio"""
3,"""Mary""",75000,"""Alto"""
4,"""Michael""",90000,"""Alto"""
5,"""Carol""",55000,"""Medio"""


# 📝 Escenario: "Reporte de Calidad de Órdenes"
Trabajamos con la tabla sales_orders.csv. El departamento de logística necesita etiquetar las órdenes para saber cuáles deben ser revisadas.

Las reglas son:

1. Filtro base: Solo queremos órdenes de los productos con ID 101, 102 y 105.
2. Nueva Columna (prioridad):
    - Si el orderstatus es 'Shipped' Y la quantity es mayor a 1, la prioridad es 'Alta'.
    - Si el billaddress es nulo o está vacío, la prioridad es 'Crítica' (porque no sabremos a quién cobrarle).
    - Para todo lo demás, la prioridad es 'Normal'.

```SQL
SELECT
    orderid,
    productid,
    orderstatus,
    CASE
        WHEN quantity > 1 AND LOWER(orderstatus) = 'shipped' THEN 'Alta'
        WHEN billaddress IS NULL OR TRIM(billaddress) = '' THEN 'Crítica'
        ELSE 'Normal'
    END AS prioridad
FROM sales.orders
WHERE productid IN (101,102,105);
```

In [24]:
df_o = df_orders.copy()

df_o = df_o[
    (df_o['productid'].isin([101,102,105]))
]

condiciones = [
    ((df_o['quantity'] > 1) & (df_o['orderstatus'].str.lower() == 'shipped')),
    ((df_o['billaddress'].isna()) | (df_o['billaddress'].str.strip() == ''))
]

elecciones = ['Alta', 'Critica']

df_o['prioridad'] = np.select(condiciones, elecciones, default='Normal')

respuesta = df_o[['orderid','productid','orderstatus','prioridad']]

respuesta


,orderid,productid,orderstatus,prioridad
0,1,101,Delivered,Normal
1,2,102,Shipped,Critica
2,3,101,Delivered,Normal
3,4,105,Shipped,Alta
6,7,102,Delivered,Critica
7,8,101,Shipped,Alta
8,9,101,Shipped,Alta
9,10,102,Shipped,Critica


In [30]:
respuesta_pl = pl_orders.filter(
    pl.col('productid').is_in([101,102,105])
).with_columns(
    pl.when(
        (pl.col('quantity') > 1) &
        (pl.col('orderstatus').str.to_lowercase() == 'shipped')
    ).then(pl.lit('Alta'))
    .when(
        (pl.col('billaddress').is_null()) |
        (pl.col('billaddress').str.strip_chars() == '')
    ).then(pl.lit('Critica'))
    .otherwise(pl.lit('Normal'))
    .alias('prioridad')
).select(['orderid', 'productid', 'orderstatus', 'prioridad'])

respuesta_pl

orderid,productid,orderstatus,prioridad
i64,i64,str,str
1,101,"""Delivered""","""Normal"""
2,102,"""Shipped""","""Critica"""
3,101,"""Delivered""","""Normal"""
4,105,"""Shipped""","""Alta"""
7,102,"""Delivered""","""Critica"""
8,101,"""Shipped""","""Alta"""
9,101,"""Shipped""","""Alta"""
10,102,"""Shipped""","""Critica"""


# 📝 Escenario: "Auditoría de Clientes Estratégicos"
Vamos a trabajar con la tabla sales_customers.csv. El departamento de fidelización quiere identificar clientes para una campaña de marketing.

Requerimientos:

1. Filtro: Solo queremos clientes de 'USA' o 'Germany'.
2. Nueva Columna (Estatus_Score):
    - Si el score es NULL, la etiqueta es 'Sin Registro'.
    - Si el score es mayor o igual a 500, la etiqueta es 'Premium'.
    - Para cualquier otro valor de score, la etiqueta es 'Estándar'.
3. Selección final: Solo necesitamos el firstname, el country y la nueva columna Estatus_Score.
4. Alias: La columna firstname debe aparecer como 'Nombre_Cliente'.

```SQL
SELECT
    firstname AS Nombre_Cliente,
    country,
    CASE
        WHEN score IS NULL THEN 'Sin Registror'
        WHEN score >= 500 THEN 'Premium'
        ELSE 'Estándar'
    END AS Estatus_Score
FROM sales.customers
WHERE LOWER(country) IN ('usa','germany');
```

In [36]:
df_c = df_customers.copy()

df_c = df_c[
    (df_c['country'].str.lower().isin(['usa','germany']))
]

condiciones = [
    (df_c['score'].isna()),
    (df_c['score'] >= 500)
]

elecciones = ['Sin Registro','Premium']

df_c['Estatus_Score'] = np.select(condiciones, elecciones, default='Estandar')

respuesta = df_c[['firstname','country','Estatus_Score']]

In [ ]:
respuesta_pl = pl_customers.filter(
    pl.col('country').str.to_lowercase().is_in(['usa','germany'])
).with_columns(
    pl.when(pl.col('score').is_null())
        .then(pl.lit('Sin Registro'))
    .when(pl.col('score') >= 500)
        .then(pl.lit('Premium'))
    .otherwise(pl.lit('Estándar'))
    .alias('Estatus_Score')    
).select(
    pl.col('firstname').alias('Nombre_Cliente'),
    pl.col('country'),
    pl.col('Estatus_Score')
)

respuesta_pl

Nombre_Cliente,country,Estatus_Score
str,str,str
"""Jossef""","""Germany""","""Estándar"""
"""Kevin""","""USA""","""Premium"""
"""Mary""","""USA""","""Premium"""
"""Mark""","""Germany""","""Premium"""
"""Anna""","""USA""","""Sin Registro"""


In [ ]:
respuesta_pl = pl_orders.filter(
    pl.col('productid').is_in([101,102,105])
).with_columns(
    pl.when(
        (pl.col('quantity') > 1) &
        (pl.col('orderstatus').str.to_lowercase() == 'shipped')
    ).then(pl.lit('Alta'))
    .when(
        (pl.col('billaddress').is_null()) |
        (pl.col('billaddress').str.strip_chars() == '')
    ).then(pl.lit('Critica'))
    .otherwise(pl.lit('Normal'))
    .alias('prioridad')
).select(['orderid', 'productid', 'orderstatus', 'prioridad'])